# Домашнее задание: Полноценная мультиагентная система с инструментами

Цель: спроектировать и реализовать систему из нескольких LLM‑агентов, которые координируются, обмениваются сообщениями и вызывают инструменты для решения реальной задачи end‑to‑end.


## Требования (обязательные)
- Роли агентов (минимум 3) с чётко разделёнными обязанностями и разными наборами инструментов для каждой роли.
- Коммуникация: шина сообщений или адресные сообщения (ask_agent / reply) с логами входящих/исходящих сообщений.
- Инструменты (минимум 3) с реальными побочными эффектами: например, работа с файлами/кодом/HTTP/данными/оценкой.
- SGR/Structured Output: схемы действий/сообщений (UseTool/AskAgent/Reply/Finish) с валидацией.
- Оркестрация: централизованный планировщик или децентрализованная логика (обосновать выбор).
- Демонстрация: один целевой сценарий, доведённый до finish, с логами и артефактами (код/отчёт/файлы).


## Оценивание (10 баллов)
- **3 балла — Дизайн агентов и протокола**
  - Чёткие роли и разграничение ответственности (1)
  - Разные доступные инструменты у разных ролей (1)
  - SGR/схемы и валидация structured output (1)
- **5 баллов — Мультиагентность и взаимодействие**
  - Координация и обмен сообщениями между агентами (2)
  - Реальные вызовы инструментов и побочные эффекты (2)
  - Обработка ошибок/ретраи/эскалация (1)
- **1 балл — Качество кода и отчёта**
  - Читаемость, структура, документация и воспроизводимость
- **1 балл — Бонус: >5 уникальных инструментов**
  - Разные по назначению функции, а не дубликаты


## Идеи задач
- Кодогенерация по спецификации: Planner → Coder → Tester → Reviewer.
- Data/RAG пайплайн: Ingestor → Indexer → Analyst → Evaluator.
- Веб‑интеграция: Researcher (web/http), Summarizer, Reporter, Verifier.

## Как сдавать
- Ноутбук с кодом агентов, инструментов и оркестратора.
- README с архитектурой (роли, взаимодействие), списком инструментов, схемами SGR, логами одного прогона, ограничениями и метриками.
- (Опционально) Короткое видео/гиф ≤3 мин с успешным прогоном.

## Штрафы
- − до 2: нет логов или невоспроизводимо
- − до 2: инструменты не делают реальных действий
- − до 1: смешение ролей/инструментов без обоснования


## Стартовые подсказки (не обязательно)
- Начните с 3 ролей и простого набора инструментов; затем добавляйте Reviewer/Verifier/Deployer.
- Схемы SGR: определите классы UseTool/AskAgent/Reply/Finish.
- Ограничьте инструменты по ролям на уровне оркестратора (enforcement).
- Логи: печатайте inbox/outbox, действие (JSON), результаты инструментов (stdout/error), diff при изменении файлов.
- Введите ретраи и сообщения «поясни ошибку», если JSON от модели невалиден или инструмент упал.


In [1]:
!pip install -q python-dotenv

In [2]:
from dotenv import load_dotenv

load_dotenv() 

True

In [4]:
# 1) Настройка OpenRouter (обязательное использование LLM)

import os
from typing import Optional

#os.environ["OPENROUTER_API_KEY"]="Your key here"

DEFAULT_MODEL = "qwen/qwen3-30b-a3b-instruct-2507"  # можно поменять на совместимую с JSON schema
openrouter_client = None

if os.environ.get("OPENROUTER_API_KEY"):
    try:
        from openai import OpenAI
        openrouter_client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ["OPENROUTER_API_KEY"],
        )
        print("OpenRouter готов. Модель:", DEFAULT_MODEL)
    except Exception as e:
        raise RuntimeError(f"Не удалось инициализировать OpenRouter: {e}")
else:
    raise RuntimeError("OPENROUTER_API_KEY не найден. Установите ключ в окружении: export OPENROUTER_API_KEY=...")


OpenRouter готов. Модель: qwen/qwen3-30b-a3b-instruct-2507


In [5]:
# 2) вызов LLM с JSON Schema

from pydantic import BaseModel
from typing import Optional
import json

def call_llm_with_schema(
    prompt: str,
    response_schema: BaseModel,
    system_prompt: Optional[str] = None
) -> BaseModel:
    if openrouter_client is None:
        raise RuntimeError("OpenRouter клиент не инициализирован. Установите OPENROUTER_API_KEY и перезапустите kernel.")
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    schema_dict = response_schema.model_json_schema()
    resp = openrouter_client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": response_schema.__name__, "schema": schema_dict, "strict": True},
        },
        temperature=0.2,
        max_tokens=1500,
    )
    raw = resp.choices[0].message.content or ""
    
    def extract_json(s: str) -> str:
        s = s.strip()
        if s.startswith("```"):
            lines = s.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            s = "\n".join(lines)
        s = s.strip()
        try:
            json.loads(s)
            return s
        except Exception:
            pass
        first = s.find('{'); last = s.rfind('}')
        if first != -1 and last != -1 and last > first:
            cand = s[first:last+1]
            try:
                json.loads(cand)
                return cand
            except Exception:
                return s
        return s
    parsed = extract_json(raw)
    return response_schema.model_validate_json(parsed)


In [6]:
# 3) SGR: действия агента и сообщения

from typing import Union, Dict, Any, List, Literal
from pydantic import BaseModel, Field

class AskAgent(BaseModel):
    action: str = "ask_agent"
    target: Literal["planner", "coder", "tester", "reviewer", "researcher"]
    message: str

class UseTool(BaseModel):
    action: str = "use_tool"
    tool_name: str  # валидация будет на уровне оркестратора
    args: Dict[str, Any]

class Reply(BaseModel):
    action: str = "reply"
    content: str

class Finish(BaseModel):
    action: str = "finish"
    summary: str

class AgentAction(BaseModel):
    step: Union[AskAgent, UseTool, Reply, Finish]

class BusMessage(BaseModel):
    sender: str
    recipient: str  # конкретная роль/агент или "broadcast"
    content: str


In [7]:
# 4) Workspace и инструменты (минимальный набор + шаблоны)

from io import StringIO
import sys
import traceback
import difflib

class Workspace:
    def __init__(self):
        self.files: Dict[str, str] = {}
        self.memory: Dict[str, Any] = {}

    # Базовые инструменты
    def store_code(self, filename: str, code: str) -> Dict[str, Any]:
        prev = self.files.get(filename, "")
        self.files[filename] = code
        diff = "\n".join(difflib.unified_diff(prev.splitlines(), code.splitlines(), fromfile=f"prev:{filename}", tofile=f"new:{filename}", lineterm=""))
        return {"message": f"stored {filename}", "filename": filename, "chars": len(code), "diff": diff[:2000]}

    def read_code(self, filename: str) -> Dict[str, Any]:
        return {"filename": filename, "code": self.files.get(filename, "")}

    def run_python(self, code: str) -> Dict[str, Any]:
        captured = StringIO()
        old = sys.stdout
        sys.stdout = captured
        out = {"stdout": "", "error": ""}
        try:
            allowed = {"print": print, "range": range, "len": len, "sum": sum, "min": min, "max": max, "abs": abs}
            exec_globals = {"__builtins__": allowed}
            exec_locals = self.memory
            exec(code, exec_globals, exec_locals)
            out["stdout"] = captured.getvalue()
        except Exception:
            out["error"] = traceback.format_exc()
        finally:
            sys.stdout = old
        return out

    def run_tests(self, filename: str, tests_code: str) -> Dict[str, Any]:
        code = self.files.get(filename, "")
        combined = code + "\n\n" + tests_code
        res = self.run_python(combined)
        return {"passed": res["error"] == "", **res}

# Регистр инструментов (измените под свои инструменты)
#class Tools:
#    def __init__(self, ws: Workspace):
#        self.ws = ws
#
#    def call(self, tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
#        args = args or {}
#        # Шаблоны под дополнительные инструменты (>5):
#        if name == "store_code":
#            filename = args.get("filename", "solution.py")
#            code = args.get("code", "")
#            if not code:
#                return {"error": "store_code requires 'code'", "required": ["filename","code"]}
#            prev = self.ws.read_code(filename)
#            msg = self.ws.store_code(filename, code)
#            diff = "\n".join(
#                difflib.unified_diff(
#                    prev.splitlines(), code.splitlines(),
#                    fromfile=f"prev:{filename}", tofile=f"new:{filename}", lineterm=""
#                )
#            )
#            head = code.splitlines()[:20]
#            return {
#                "message": msg,
#                "filename": filename,
#                "chars": len(code),
#                "head": "\n".join(head),
#                "diff": diff[:4000]
#            }
#        if tool_name == "summarize_text":
#            text = args.get("text", "")
#            return {"summary": text[:200], "info": "stub: implement real summarizer or LLM call"}
#        if tool_name == "transform_data":
#            data = args.get("data", [])
#            return {"size": len(data), "preview": str(data)[:200]}
#        if tool_name == "plan_schedule":
#            tasks = args.get("tasks", [])
#            return {"plan": [f"step-{i+1}: {t}" for i, t in enumerate(tasks[:10])]} 
#        return {"error": f"unknown tool {tool_name}"}

class Tools:
    def __init__(self, ws: Workspace):
        self.ws = ws

    def call(self, tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
        args = args or {}

        # ===============================
        # ⚙️ Основные инструменты Workspace
        # ===============================

        if tool_name == "store_code":
            return self.ws.store_code(
                filename=args.get("filename"),
                code=args.get("code") or args.get("content") or ""
            )

        if tool_name == "read_code":
            return self.ws.read_code(
                filename=args.get("filename")
            )

        if tool_name == "run_python":
            return self.ws.run_python(
                code=args.get("code", "")
            )

        if tool_name == "run_tests":
            filename = args.get("filename")
            tests_code = args.get("tests_code", "")
            return self.ws.run_tests(filename, tests_code)

        # ===============================
        # 🌐 Прочие инструменты (stub)
        # ===============================

        if tool_name == "summarize_text":
            text = args.get("text", "")
            return {"summary": text[:200]}

        if tool_name == "transform_data":
            data = args.get("data", [])
            return {"size": len(data), "preview": str(data)[:200]}

        if tool_name == "plan_schedule":
            tasks = args.get("tasks", [])
            return {"plan": [f"step-{i+1}: {t}" for i, t in enumerate(tasks[:10])]} 

        # Fallback
        return {"error": f"unknown tool {tool_name}"}


ws = Workspace()
tools = Tools(ws)



In [8]:
# 5) Шина сообщений и шаблон агента

from collections import deque

class MessageBus:
    def __init__(self):
        self.queue = deque()
        self.history: List[BusMessage] = []
    def send(self, msg: BusMessage):
        self.queue.append(msg); self.history.append(msg)
    def receive_for(self, recipient: str) -> List[BusMessage]:
        msgs = [m for m in list(self.queue) if m.recipient in (recipient, "broadcast")]
        for m in msgs:
            try:
                self.queue.remove(m)
            except ValueError:
                pass
        return msgs

bus = MessageBus()

class LlmAgent:
    def __init__(self, name: str, system_role: str, allowed_tools: List[str]):
        self.name = name
        self.system_role = system_role
        self.allowed_tools = allowed_tools
    
    def decide(self, goal: str) -> AgentAction:
        """Сформируйте промпт и позовите call_llm_with_schema. Верните AgentAction."""
        history_text = "\n".join([f"{m.sender}→{m.recipient}: {m.content}" for m in bus.history][-10:])
        tools_str = ", ".join(self.allowed_tools) if self.allowed_tools else "(нет)"
        prompt = f"""
Вы — {self.system_role}. Ваша роль: {self.name}.
Цель: {goal}
История:
{history_text}

Выберите одно действие:
- ask_agent (target, message)
- use_tool (tool_name in [{tools_str}], args)
- reply (content)
- finish (summary)

Интерфейс инструментов (STRICT):
- store_code args: {{"filename": str (default: "solution.py"), "code": str (required)}}
- read_code  args: {{"filename": str (default: "solution.py")}}
- run_python args: {{"code": str (required)}}
- run_tests  args: {{"filename": str (default: "solution.py"), "tests_code": str (default: tests variable)}}

ОГРАНИЧЕНИЯ:
- Нельзя вызывать инструменты, которых нет в списке ДЛЯ ТЕКУЩЕЙ РОЛИ: {tools_str}
- Если вы tester: после успешных тестов (passed=true) выберите finish с кратким резюме.
- Если инструмент вернул error, сформулируйте ask_agent к соответствующей роли с подробностями ошибки.


Верните только JSON по схеме AgentAction.
"""
        return call_llm_with_schema(prompt=prompt, response_schema=AgentAction, system_prompt=self.system_role)


In [9]:
# 6) Оркестратор (скелет)

class Orchestrator:
    def __init__(self):
        self.agents = {
            "planner":  LlmAgent("planner",  "Планировщик", []),
            "coder":    LlmAgent("coder",    "Разработчик", ["store_code", "read_code", "run_python"]),
            "tester":   LlmAgent("tester",   "Тестировщик", ["read_code", "run_tests"]),
            # добавляйте свои роли, например reviewer/researcher
        }
        self.allowed_tools = {name: a.allowed_tools for name, a in self.agents.items()}
    
    def step(self, goal: str, order: List[str]) -> bool:
        for name in order:
            inbox = bus.receive_for(name)
            if inbox:
                print(f"[{name}] inbox ({len(inbox)}):")
                for m in inbox[-3:]:
                    print(f"  {m.sender}→{m.recipient}: {m.content[:200]}")
            action = self.agents[name].decide(goal)
            s = action.step
            print(f"[{name}] action: {s}")
            if isinstance(s, AskAgent):
                bus.send(BusMessage(sender=name, recipient=s.target, content=s.message))
                print(f"[{name}] → [{s.target}] ask: {s.message}")
            elif isinstance(s, UseTool):
                if s.tool_name not in self.allowed_tools.get(name, []):
                    msg = f"роль {name} не имеет доступа к инструменту {s.tool_name}. Доступны: {self.allowed_tools.get(name, [])}"
                    print(f"[{name}] denied {s.tool_name}: {msg}")
                    bus.send(BusMessage(sender="orchestrator", recipient=name, content=msg))
                    continue
                result = tools.call(s.tool_name, s.args)
                print(f"[{name}] used {s.tool_name} → {str(result)[:300]}")
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"tool {s.tool_name} result: {result}"))
                if s.tool_name == "run_tests" and isinstance(result, dict) and result.get("passed") is True:
                    bus.send(BusMessage(sender="tester", recipient="broadcast", content="FINISH: tests passed"))
                    return True
            elif isinstance(s, Reply):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=s.content))
                print(f"[{name}] reply: {s.content}")
            elif isinstance(s, Finish):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"FINISH: {s.summary}"))
                print(f"[{name}] finish: {s.summary}")
                return True
        return False
    
    def run(self, goal: str, max_rounds: int = 8):
        print("=== ORCHESTRATION START ===")
        bus.send(BusMessage(sender="planner", recipient="broadcast", content=f"start: {goal}"))
        for r in range(1, max_rounds+1):
            print(f"\n--- ROUND {r} ---")
            if self.step(goal, ["planner", "coder", "tester"]):
                print("=== ORCHESTRATION FINISHED ===")
                break
        else:
            print("=== MAX ROUNDS REACHED ===")


In [10]:
# 1) Задайте цель мультиагентной системы (строка)
# Примеры: "Сформировать отчёт по исследованию", "Спланировать эксперименты", "Проверить качество артефактов".
# goal = "ВАША_ЦЕЛЬ_ЗДЕСЬ"

# 2) (Опционально) Отправьте начальное сообщение в шину
# bus.send(BusMessage(sender="planner", recipient="broadcast", content=f"start: {goal}"))

# 3) (Опционально) При необходимости добавьте собственные инструменты в класс Tools
#    и расширьте allowed_tools для ролей в Orchestrator

# 4) Запустите оркестратор
# orch = Orchestrator()
# orch.run(goal=goal, max_rounds=8)

# 5) Выведите артефакты/логи из Workspace
# print("files:", list(ws.files.keys()))
# print("history:")
# for m in bus.history[-10:]:
#     print(f"{m.sender}→{m.recipient}: {m.content[:160]}")


In [11]:
#goal = (
#    "Реализовать функцию fib(n) -> int, возвращающую n-е число Фибоначчи (0-indexed), и пройти тесты.\n"
#    "Ожидается простая и эффективная реализация без избыточной памяти."
#)
#
## Подскажем участникам стартовое сообщение с подсказкой про инструменты
#bus.send(BusMessage(sender="planner", recipient="broadcast", content=(
#    "Пожалуйста, Coder: используйте use_tool/store_code чтобы записать решение в solution.py \n"
#    "Tester: используйте use_tool/run_tests с тестовым кодом. Если не проходят — сообщите и попросите доработку."
#)))
#
#orc = Orchestrator()
#orc.run(goal=goal, max_rounds=8)
#
#
#print("\nWorkspace files:", list(ws.files.keys()))
#if "solution.py" in ws.files:
#    print("solution.py snippet:\n", ws.files["solution.py"][:200])

### Необязательно использовать этот шаблон, можете написать свой!


In [12]:
import os
import json
import difflib
import traceback
from typing import Any, Dict, List, Optional, Union, Literal
from collections import deque
from pydantic import BaseModel, Field
from io import StringIO
import sys
import time

# ---------------------------
# Цветное логирование
# ---------------------------
class Log:
    """
    Утилита для цветного логирования сообщений в консоль.

    Особенности:
    ------------
    - Поддерживает уровни сообщений: INFO, WARN, ERROR, DEBUG.
    - Цветовое оформление помогает быстро различать тип сообщения.
    - Методы статические — можно вызывать без создания экземпляра.

    Атрибуты:
    ----------
    COLORS : Dict[str, str]
        Словарь соответствия уровням сообщений ANSI-кодов цветов:
        - INFO: синий
        - WARN: жёлтый
        - ERROR: красный
        - DEBUG: зелёный
        - RESET: сброс цвета
    """

    COLORS = {
        "INFO": "\033[94m",   # синий
        "WARN": "\033[93m",   # жёлтый
        "ERROR": "\033[91m",  # красный
        "DEBUG": "\033[92m",  # зелёный
        "RESET": "\033[0m"    # сброс цвета
    }

    @staticmethod
    def info(msg: str):
        """
        Логирует информационное сообщение (INFO) с синим цветом.

        Параметры:
        -----------
        msg : str
            Текст сообщения.
        """
        print(f"{Log.COLORS['INFO']}[INFO]{Log.COLORS['RESET']} {msg}")

    @staticmethod
    def warn(msg: str):
        """
        Логирует предупреждение (WARN) с жёлтым цветом.

        Параметры:
        -----------
        msg : str
            Текст предупреждения.
        """
        print(f"{Log.COLORS['WARN']}[WARN]{Log.COLORS['RESET']} {msg}")

    @staticmethod
    def error(msg: str):
        """
        Логирует сообщение об ошибке (ERROR) с красным цветом.

        Параметры:
        -----------
        msg : str
            Текст ошибки.
        """
        print(f"{Log.COLORS['ERROR']}[ERROR]{Log.COLORS['RESET']} {msg}")

    @staticmethod
    def debug(msg: str):
        """
        Логирует отладочное сообщение (DEBUG) с зелёным цветом.

        Параметры:
        -----------
        msg : str
            Текст отладочной информации.
        """
        print(f"{Log.COLORS['DEBUG']}[DEBUG]{Log.COLORS['RESET']} {msg}")


In [13]:
# ---------------------------
# SGR: Agent actions & messaging
# ---------------------------

from typing import Any, Dict, Optional, Union, Literal
from pydantic import BaseModel, Field

class AskAgent(BaseModel):
    """
    Действие: отправить сообщение другому агенту.

    Поля:
        action  – тип действия, всегда "ask_agent".
        target  – имя агента, которому адресовано сообщение
                  (planner, coder, tester, reviewer).
        message – текст сообщения, передаваемый целевому агенту.

    Назначение:
        Используется для межагентной коммуникации в рамках шины сообщений (MessageBus).
    """
    action: Literal["ask_agent"] = "ask_agent"
    target: Literal["planner", "coder", "tester", "reviewer"]
    message: str


class UseTool(BaseModel):
    """
    Действие: запросить выполнение инструмента (tool).

    Поля:
        action    – тип действия, всегда "use_tool".
        tool_name – имя инструмента, который должен быть вызван (например: store_code, run_tests).
        args      – параметры, необходимые инструменту для выполнения (словарь).

    Назначение:
        Позволяет агентам вызывать внешние инструменты через Orchestrator.
    """
    action: Literal["use_tool"] = "use_tool"
    tool_name: str
    args: Dict[str, Any] = Field(default_factory=dict)


class Reply(BaseModel):
    """
    Действие: вернуть текстовый ответ.

    Поля:
        action  – тип действия, всегда "reply".
        content – текст ответа.

    Назначение:
        Используется, когда агент должен вернуть результат,
        не инициируя вызовы инструментов и не отправляя сообщений другим агентам.
    """
    action: Literal["reply"] = "reply"
    content: str


class Finish(BaseModel):
    """
    Действие: завершить работу агента, сообщив итог.

    Поля:
        action  – тип действия, всегда "finish".
        summary – краткое описание результата завершения.

    Назначение:
        Позволяет агенту сообщить о финальном состоянии (например: задача решена,
        тесты пройдены, работа завершена) и прекратить участие в текущей оркестрации.
    """
    action: Literal["finish"] = "finish"
    summary: str


class AgentStep(BaseModel):
    """
    Обёртка над единичным шагом агента.

    Поля:
        step – одно из возможных действий агента:
               AskAgent, UseTool, Reply или Finish.

    Назначение:
        Определяет структуру ответа агента, обеспечивая строгую типизацию его действий.
    """
    step: Union[AskAgent, UseTool, Reply, Finish]


class BusMessage(BaseModel):
    """
    Сообщение, передаваемое через MessageBus.

    Поля:
        sender    – имя агента, отправившего сообщение.
        recipient – имя агента-получателя или "broadcast" для глобальной рассылки.
        content   – текст сообщения.
        meta      – дополнительные служебные метаданные (опционально).

    Назначение:
        Используется инфраструктурой для маршрутизации коммуникаций между агентами.
    """
    sender: str
    recipient: str  # имя агента или "broadcast"
    content: str
    meta: Optional[Dict[str, Any]] = None

In [14]:
class Workspace:
    """
    Виртуальная рабочая среда многоагентной системы.

    Данный класс моделирует мини-файловую систему и безопасное окружение для 
    выполнения Python-кода. Основная задача Workspace — предоставить агентам 
    (coder, tester, reviewer) возможность:

    1. Создавать и изменять виртуальные файлы с кодом.
    2. Читать содержимое файлов.
    3. Исполнять Python-код в песочнице (sandbox) с ограниченными возможностями.
    4. Запускать тесты, комбинируя код и тестовые сценарии.
    5. Хранить внутреннее состояние (memory), если это требуется логике агентов.

    Такой подход имитирует полноценную рабочую среду программиста, но без 
    взаимодействия с реальной файловой системой. Это гарантирует безопасность 
    и предсказуемость выполнения.
    """

    def __init__(self):
        """
        Инициализирует пустую виртуальную среду.

        Атрибуты:
            files (Dict[str, str]):
                Словарь, где ключ — имя файла, значение — его текстовое содержимое.

            memory (Dict[str, Any]):
                Вспомогательное хранилище произвольных данных.
                Может использоваться агентами для запоминания промежуточных
                состояний, результатов анализа, подсказок и др.

        Важно:
            Виртуальные файлы существуют только в памяти процесса.
            На реальный диск ничего не записывается.
        """
        self.files: Dict[str, str] = {}
        self.memory: Dict[str, Any] = {}

    def store_code(self, filename: str, code: str) -> Dict[str, Any]:
        """
        Сохраняет (или перезаписывает) содержимое виртуального файла.

        Параметры:
            filename (str):
                Имя файла (например: "solution.py").
            code (str):
                Новый текст файла.

        Поведение:
            • Если файл ранее существовал — вычисляется diff (разница между старым и новым содержимым).
            • Новый код заменяет старый полностью.
            • Возвращается информация о количестве символов и вычисленный diff.

        Возвращает:
            dict:
            {
                "message": "stored <filename>",
                "filename": filename,
                "chars": <кол-во символов в новом файле>,
                "diff": <строка с unix-стилем diff>
            }

        Зачем нужен diff:
            Агент reviewer или orchestrator может использовать diff для анализа изменений,
            не перечитывая весь файл.

        Примечание:
            Это *не* запись в реальную файловую систему — всё хранится в self.files.
        """
        prev = self.files.get(filename, "")
        self.files[filename] = code

        diff = "\n".join(
            difflib.unified_diff(
                prev.splitlines(),
                code.splitlines(),
                fromfile=f"prev:{filename}",
                tofile=f"new:{filename}",
                lineterm=""
            )
        )

        Log.info(f"Файл '{filename}' сохранен, символов: {len(code)}")

        return {
            "message": f"stored {filename}",
            "filename": filename,
            "chars": len(code),
            "diff": diff
        }

    def read_code(self, filename: str) -> Dict[str, Any]:
        """
        Возвращает содержимое виртуального файла.

        Параметры:
            filename (str):
                Имя файла, который нужно прочитать.

        Возвращает:
            dict:
            {
                "filename": filename,
                "code": <текст файла или пустая строка, если файл не существует>
            }

        Примечание:
            Метод не генерирует ошибок при отсутствии файла — просто возвращает пустой код.
        """
        return {"filename": filename, "code": self.files.get(filename, "")}

    def run_python(self, code: str, timeout_seconds: float = 3.0) -> Dict[str, Any]:
        """
        Выполняет переданный Python-код в ограниченной среде (sandbox).

        Параметры:
            code (str):
                Строка с Python-кодом для выполнения.
            timeout_seconds (float):
                (Планируется для будущих версий) лимит времени на выполнение.

        Поведение:
            • Перехватывает stdout (все print выводы).
            • Разрешает только безопасный набор встроенных функций (print, range, len, int, float и т.п.).
            • Полностью запрещает импорт модулей и доступ к системным ресурсам.
            • В случае ошибки возвращает трассировку (traceback).

        Возвращает:
            dict:
            {
                "stdout": <вывод программы>,
                "error": <текст ошибки или пустая строка>
            }

        Зачем нужна песочница:
            — Исключает возможность выполнения вредоносного или опасного кода.
            — Предотвращает доступ к диску, сети и запрещённым встроенным функциям.
        """
        captured = StringIO()
        old_out = sys.stdout
        sys.stdout = captured
        out = {"stdout": "", "error": ""}

        try:
            allowed = {
                "print": print, "range": range, "len": len, "sum": sum,
                "min": min, "max": max, "abs": abs,
                "int": int, "float": float, "str": str, "bool": bool,
                "list": list, "dict": dict, "set": set, "tuple": tuple
            }

            exec_globals = {"__builtins__": allowed}
            exec_locals = {}

            exec(code, exec_globals, exec_locals)

            out["stdout"] = captured.getvalue()

        except Exception:
            out["error"] = traceback.format_exc()
            Log.error(out["error"])

        finally:
            sys.stdout = old_out

        return out

    def run_tests(self, filename: str, tests_code: str) -> Dict[str, Any]:
        """
        Выполняет тесты для указанного файла.

        Параметры:
            filename (str):
                Имя файла, код которого нужно тестировать.
            tests_code (str):
                Набор Python-тестов (чаще всего — assert выражения).

        Поведение:
            • Если файла нет — возвращает ошибку.
            • Объединяет основной код и тестовый код в одну строку.
            • Выполняет их через run_python().
            • Считает тесты пройденными, если не возникло ошибок.

        Возвращает:
            dict:
            {
                "passed": True/False,
                "stdout": <вывод программы>,
                "error": <текст ошибки или пустая строка>
            }

        Применение:
            Агент tester вызывает этот метод после генерации тестов.
            Если тесты прошли успешно — orchestrator может завершить работу системы.
        """
        code = self.files.get(filename, "")
        if not code:
            return {
                "passed": False,
                "stdout": "",
                "error": f"Файл {filename} не найден"
            }

        combined = code + "\n\n" + tests_code
        res = self.run_python(combined)
        passed = res["error"] == ""

        Log.info(f"Тесты файла '{filename}' {'пройдены' if passed else 'не пройдены'}")

        return {"passed": passed, **res}


In [15]:
class Tools:
    """
    Набор инструментов, доступных агентам системы оркестрации.

    Этот класс является адаптером между агентами (planner, coder, tester и др.)
    и внутренней виртуальной рабочей средой Workspace, предоставляя единый
    интерфейс для выполнения действий над кодом и файлами.

    Каждый инструмент вызывается через метод `call()`, который маршрутизирует
    запрос к соответствующему методу Workspace или выполняет встроенную
    функциональность (например, lint).

    Поддерживаемые инструменты и их логирование описаны внутри call().
    """

    def __init__(self, ws: Workspace):
        """
        Инициализация набора инструментов.

        Параметры:
        ----------
        ws : Workspace
            Объект виртуальной рабочей среды, предоставляющий низкоуровневые
            операции над файлами и исполняемым кодом.
        """
        self.ws = ws

    def call(self, tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
        """
        Основной интерфейс вызова инструментов.

        На основе имени инструмента вызывает соответствующие методы Workspace
        либо выполняет встроенную логику. Логирует вызовы и результаты.

        Параметры:
        ----------
        tool_name : str
            Имя вызываемого инструмента.
        args : dict
            Аргументы, необходимые для выполнения инструмента.

        Возвращает:
        -----------
        dict
            Результат работы выбранного инструмента.
        """
        args = args or {}
        Log.debug(f"[Tools] Вызов инструмента: {tool_name}, args: {args}")

        # ------------------------
        # store_code — сохранить файл
        # ------------------------
        if tool_name == "store_code":
            filename = args.get("filename", "solution.py")
            code = args.get("code") or args.get("content") or ""
            result = self.ws.store_code(filename, code)
            Log.info(f"[Tools] Файл '{filename}' сохранён, символов: {result['chars']}")
            return result

        # ------------------------
        # read_code — получить содержимое файла
        # ------------------------
        if tool_name == "read_code":
            filename = args.get("filename", "solution.py")
            result = self.ws.read_code(filename)
            Log.info(f"[Tools] Прочитан файл '{filename}', символов: {len(result.get('code',''))}")
            return result

        # ------------------------
        # run_python — выполнить произвольный код
        # ------------------------
        if tool_name == "run_python":
            code = args.get("code", "")
            result = self.ws.run_python(code)
            if result.get("error"):
                Log.error(f"[Tools] Ошибка при выполнении кода: {result['error']}")
            else:
                Log.info(f"[Tools] Код выполнен успешно, вывод: {result['stdout'][:200]}")
            return result

        # ------------------------
        # run_tests — выполнить решение + тесты
        # ------------------------
        if tool_name == "run_tests":
            filename = args.get("filename", "solution.py")
            tests_code = args.get("tests_code", "")
            result = self.ws.run_tests(filename, tests_code)
            if result.get("passed"):
                Log.info(f"[Tools] Тесты файла '{filename}' пройдены успешно.")
            else:
                Log.warn(f"[Tools] Тесты файла '{filename}' не пройдены, ошибка: {result.get('error','')[:200]}")
            return result

        # ------------------------
        # lint_code — базовый линтер
        # ------------------------
        if tool_name == "lint_code":
            code = args.get("code", "")
            issues = []
            if "    " in code and "\t" in code:
                issues.append("Смешение табуляций и пробелов")
            result = {"issues": issues, "issue_count": len(issues)}
            Log.info(f"[Tools] Линтинг кода завершён, найдено проблем: {len(issues)}")
            return result

        # ------------------------
        # generate_tests — базовый генератор тестов
        # ------------------------
        if tool_name == "generate_tests":
            func_name = args.get("func_name", "solution")
            tests = (
                "assert fib(0) == 0\n"
                "assert fib(1) == 1\n"
                "assert fib(5) == 5\n"
                "assert fib(10) == 55\n"
                "print('Тесты пройдены')\n"
            )
            Log.info(f"[Tools] Сгенерированы тесты для функции '{func_name}'")
            return {"tests_code": tests}

        # ------------------------
        # неизвестный инструмент
        # ------------------------
        error_msg = f"неизвестный инструмент {tool_name}"
        Log.error(f"[Tools] {error_msg}")
        return {"error": error_msg}



In [16]:
# ---------------------------
# Шина сообщений
# ---------------------------
class MessageBus:
    """
    Шина обмена сообщениями между агентами многоагентной системы.

    MessageBus — это центральный коммуникационный механизм, через который
    агенты обмениваются сообщениями. Он выполняет сразу несколько функций:

    1. Буферизация сообщений  
       Сообщения помещаются в FIFO-очередь (`deque`) и извлекаются агентами
       по мере обработки.

    2. Адресная доставка сообщений  
       Каждое сообщение содержит поле `recipient`, которое может быть именем
       одного конкретного агента либо строкой "broadcast".  
       В случае бродкаста сообщение будет получено каждым агентом.

    3. Ведение истории  
       Все отправленные сообщения сохраняются в списке `history`, что
       позволяет:
       • анализировать ход работы системы  
       • выполнять отладку  
       • строить визуализацию взаимодействий

    4. Логирование  
       Каждый вызов send() создает запись в журнале Log.debug, упрощая трассировку
       обмена агентами.

    Структура данных:
    -----------------
    queue : deque  
        Очередь сообщений, ожидающих доставки агенту.

    history : list[BusMessage]  
        Полный журнал всех отправленных сообщений (не только необработанных).

    Рабочий цикл:
    -------------
    • Агент вызывает message_bus.receive_for("coder")  
      → получает список всех сообщений, адресованных ему или broadcast.

    • Все извлечённые сообщения удаляются из очереди, но остаются в history.

    Ограничения:
    ------------
    • Порядок сообщений сохраняется.  
    • Шина не гарантирует atomicity или параллелизм — это простой механизм
      обмена между шагами оркестрации.  
    • Сообщения не доставляются повторно: после выдачи receive_for() они удаляются.

    """

    def __init__(self):
        """
        Инициализация шины сообщений.

        Создаёт пустую очередь для входящих сообщений и хранилище истории.

        queue  — основная очередь сообщений  
        history — список всех отправленных сообщений для аудита
        """
        self.queue = deque()
        self.history: List[BusMessage] = []

    def send(self, msg: BusMessage):
        """
        Отправляет сообщение в шину.

        Сообщение добавляется в очередь, а его копия фиксируется в истории.
        Также генерируется запись в лог (Log.debug).

        Параметры:
        ----------
        msg : BusMessage
            Сообщение, включающее отправителя, получателя и содержимое.

        Поведение:
        ----------
        • Помещает msg в очередь queue.  
        • Добавляет msg в history.  
        • Логирует отправку с сокращением длинного текста до 200 символов.

        """
        self.queue.append(msg)
        self.history.append(msg)
        Log.debug(f"{msg.sender} → {msg.recipient}: {str(msg.content)[:200]}")

    def receive_for(self, recipient: str) -> List[BusMessage]:
        """
        Извлекает все сообщения, предназначенные указанному агенту.

        Это может быть:
        • адресная доставка (msg.recipient == recipient)
        • широковещательная доставка (msg.recipient == "broadcast")

        После выдачи сообщения удаляются из очереди, но остаются в history.

        Параметры:
        ----------
        recipient : str
            Имя агента (например: "planner", "coder", "tester")

        Возвращает:
        -----------
        List[BusMessage]
            Список всех сообщений, ожидавших данного получателя.

        Примечания:
        -----------
        • Удаление выполняется осторожно: если кто-то уже забрал сообщение,
          ValueError игнорируется.

        • Порядок сообщений сохраняется, так как используется копия queue для фильтрации.
        """
        msgs = [m for m in list(self.queue) if m.recipient in (recipient, "broadcast")]

        for m in msgs:
            try:
                self.queue.remove(m)
            except ValueError:
                # Сообщение могло быть удалено параллельным вызовом receive_for()
                pass

        return msgs


In [17]:
# ---------------------------
# LLM caller with JSON schema; retries on invalid JSON
# If OpenRouter is not available, fall back to deterministic mock agent behavior.
# ---------------------------

def call_llm_with_schema(prompt: str, response_model: BaseModel, system_prompt: Optional[str] = None,
                         max_retries: int = 3, temperature: float = 0.2) -> BaseModel:
    """
    Универсальный вызов LLM с жёсткой JSON-схемой и механизмом повторных попыток.

    Функция отправляет запрос к модели через openrouter_client и ожидает,
    что LLM вернёт корректный JSON, соответствующий заданной Pydantic-схеме
    (response_model).  

    Возможности:
    ------------
    1. **Строгое требование JSON по схеме**
       LLM получает инструкцию вернуть JSON-объект, соответствующий
       модели response_model.

    2. **Автоматические повторы при ошибках**
       Если LLM вернул непарсибельный JSON или объект, не соответствующий схеме,
       выполняется до `max_retries` повторов с уточняющими сообщениями.

    3. **Fallback, если OpenRouter недоступен**
       Если openrouter_client = None, вызов полностью переключается
       на локальный детерминированный мок — fallback_llm(), чтобы можно было
       тестировать логику агентов без интернета.

    4. **Автоматическая очистка от ```json ... ```**
       Функция _extract_json_from_text() пытается извлечь JSON даже если
       LLM вернул текст с Markdown-оформлением.

    Параметры:
    ----------
    prompt : str
        Основной пользовательский запрос для LLM.

    response_model : BaseModel
        Pydantic-модель, описывающая структуру ожидаемого результата.
        LLM обязан вернуть JSON, который ей соответствует.

    system_prompt : Optional[str]
        Системная подсказка, управляющая поведением LLM.

    max_retries : int
        Максимальное количество попыток получить валидный JSON.

    temperature : float
        "Температура" генерации модели (0–1). Чем меньше — тем более детерминированный вывод.

    Возвращает:
    -----------
    BaseModel
        Экземпляр response_model, успешно извлечённый из ответа LLM.

    Исключения:
    -----------
    RuntimeError  
        Если все попытки завершились невалидным JSON.

    Комментарии:
    ------------
    При неудаче добавляет в сообщения запрос "Пожалуйста, верни строго JSON…",
    что помогает направить LLM к корректному выводу.
    """
    Log.debug(f"[LLM] Вызов с prompt: {prompt[:200]}, system_prompt: {system_prompt}, max_retries: {max_retries}")

    if openrouter_client is None:
        Log.warn("[LLM] OpenRouter не доступен, используется fallback_llm")
        return fallback_llm(prompt, response_model, system_prompt)

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    schema_dict = response_model.model_json_schema()
    last_raw = ""

    for attempt in range(1, max_retries + 1):
        try:
            Log.debug(f"[LLM] Попытка {attempt}")
            resp = openrouter_client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=messages,
                response_format={
                    "type": "json_schema",
                    "json_schema": {"name": response_model.__name__, "schema": schema_dict, "strict": True},
                },
                temperature=temperature,
                max_tokens=1500,
            )
            raw = resp.choices[0].message.content or ""
            last_raw = raw
            parsed = _extract_json_from_text(raw)
            Log.info(f"[LLM] Успешно получен JSON на попытке {attempt}")
            return response_model.model_validate_json(parsed)

        except Exception as e:
            Log.error(f"[LLM] Попытка {attempt} не удалась: {e}")
            messages.append({
                "role": "user",
                "content": "Пожалуйста, верни строго JSON по указанной схеме, без пояснений."
            })
            time.sleep(0.3)
            continue

    Log.error(f"[LLM] Все {max_retries} попыток не удались, последний ответ:\n{last_raw}")
    raise RuntimeError(
        f"LLM did not return valid JSON after {max_retries} attempts. Last raw:\n{last_raw}"
    )


def _extract_json_from_text(s: str) -> str:
    """
    Пытается корректно извлечь JSON даже из обёрнутого в Markdown текста.

    LLM иногда возвращает результат в формате:

        ```json
        { ... }
        ```

    или даже с произвольным текстом до и после JSON.  
    Эта функция:

    1. Удаляет блоковые кавычки ``` ... ``` если они есть  
    2. Проверяет, является ли строка валидным JSON  
    3. Если нет — пытается вырезать подстроку между первой '{' и последней '}'  
    4. Если всё равно не удалось — возвращает оригинальный текст для последующей ошибки

    Параметры:
    ----------
    s : str
        Строка, которая может содержать JSON в любом виде.

    Возвращает:
    -----------
    str  
        Строка, которая, вероятно, является валидным JSON.
        (Парсинг JSON будет выполнен вызывающим кодом.)
    """
    s = (s or "").strip()
    Log.debug(f"[LLM] Извлечение JSON из текста длиной {len(s)}")

    if s.startswith("```"):
        lines = s.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        s = "\n".join(lines).strip()
        Log.debug(f"[LLM] Удалены тройные кавычки, длина после очистки: {len(s)}")

    try:
        json.loads(s)
        return s
    except Exception:
        pass

    first = s.find('{')
    last = s.rfind('}')
    if first != -1 and last != -1 and last > first:
        cand = s[first:last+1]
        try:
            json.loads(cand)
            Log.debug(f"[LLM] JSON успешно извлечён из подстроки")
            return cand
        except Exception:
            pass

    Log.warn("[LLM] Не удалось извлечь корректный JSON, возвращается оригинал")
    return s


def fallback_llm(prompt: str, response_model: BaseModel, system_prompt: Optional[str]):
    """
    Детерминированный локальный мок LLM для автономного тестирования системы.

    Данный режим используется, если openrouter_client == None.
    То есть система работает полностью оффлайн, но логика агентов остаётся.

    Задачи fallback-режима:
    ----------------------
    • позволить тестировать мультиагентную архитектуру без доступа в интернет  
    • вернуть валидный JSON, соответствующий response_model  
    • имитировать поведение агентов на основе простых эвристик  

    Поведение:
    ----------
    • Если в prompt упоминается store_code / write code / implement → формируется UseTool
    • Если встречается "fib" → генерируется код решения задачи FIB
    • Если встречаются слова test / tests → генерируем команду generate_tests
    • В остальных случаях — обычный Reply("ack")

    Параметры:
    ----------
    prompt : str
        Текстовый запрос "от пользователя".

    response_model : BaseModel
        Модель, которую необходимо вернуть.

    system_prompt : Optional[str]
        (не используется, но передаётся)

    Возвращает:
    -----------
    BaseModel
        Заполненная структура, строго соответствующая JSON-схеме.
    """
    text = (prompt or "").lower()
    Log.debug(f"[LLM fallback] Вызов с prompt: {prompt[:200]}")

    if "store_code" in text or "write code" in text or "implement" in text or "fib" in text:
        if "fib" in text:
            code = (
                "def fib(n: int) -> int:\n"
                "    if n <= 1:\n"
                "        return n\n"
                "    a, b = 0, 1\n"
                "    for _ in range(2, n + 1):\n"
                "        a, b = b, a + b\n"
                "    return b\n"
            )
            obj = {
                "step": {
                    "action": "use_tool",
                    "tool_name": "store_code",
                    "args": {"filename": "solution.py", "code": code}
                }
            }
            Log.info("[LLM fallback] Генерируем код FIB для store_code")
            return response_model.model_validate_json(json.dumps(obj))

        obj = {"step": {"action": "reply", "content": "Please provide code or clarify requirements."}}
        Log.info("[LLM fallback] Генерируем fallback reply для coder")
        return response_model.model_validate_json(json.dumps(obj))

    if "run_tests" in text or "tests" in text or "test" in text:
        obj = {
            "step": {
                "action": "use_tool",
                "tool_name": "generate_tests",
                "args": {"func_name": "fib"}
            }
        }
        Log.info("[LLM fallback] Генерируем команду generate_tests для tester")
        return response_model.model_validate_json(json.dumps(obj))

    obj = {"step": {"action": "reply", "content": "ack"}}
    Log.info("[LLM fallback] Генерируем default reply ack")
    return response_model.model_validate_json(json.dumps(obj))


In [18]:
# ---------------------------
# Базовый агент и конкретные агенты
# ---------------------------

class AgentBase:
    """
    Базовый класс агента.

    Предназначен для всех агентов мультиагентной системы (planner, coder, tester, reviewer). 
    Содержит логику формирования запроса к LLM, обработки входящих сообщений и выбора действия.

    Атрибуты:
    ----------
    name : str
        Уникальное имя агента (например, "coder").

    system_role : str
        Описание роли агента, используется при формировании системного промпта для LLM.

    allowed_tools : List[str]
        Список инструментов (tool_name), которыми агент может пользоваться.
        Например: ["store_code", "read_code", "run_python"].

    Методы:
    --------
    decide(goal: str, inbox: List[BusMessage]) -> AgentStep
        Определяет действие агента на основе текущей цели и входящих сообщений.
        Возвращает экземпляр AgentStep, строго соответствующий Pydantic-схеме.
    """

    def __init__(self, name: str, system_role: str, allowed_tools: List[str]):
        self.name = name
        self.system_role = system_role
        self.allowed_tools = allowed_tools

    def decide(self, goal: str, inbox: List[BusMessage]) -> AgentStep:
        """
        Принимает решение о следующем действии агента.

        Параметры:
        -----------
        goal : str
            Текущая цель или задача, которую агент должен решить.
        inbox : List[BusMessage]
            Список сообщений, адресованных агенту. Используется для анализа последних
            взаимодействий и контекста.

        Логика:
        -------
        1. Формируется текстовый prompt для LLM, включающий:
            - роль агента и системное описание
            - текущую цель
            - последние сообщения (inbox)
            - история последних 20 сообщений в шине
            - список доступных инструментов

        2. Вызов LLM через call_llm_with_schema, с валидацией результата через Pydantic.

        3. Если LLM возвращает некорректный JSON или выбрасывает ошибку:
            - логируется ошибка
            - возвращается объект AgentStep с действием reply, информирующий
              о проблеме (fallback для безопасности).

        Возвращает:
        -----------
        AgentStep
            Экземпляр Pydantic-модели, описывающий выбранное действие:
            - AskAgent (задать вопрос другому агенту)
            - UseTool (вызвать инструмент из allowed_tools)
            - Reply (простое сообщение в шину)
            - Finish (сообщение о завершении задачи)
        """
        hist = "\n".join([f"{m.sender}->{m.recipient}: {m.content}" for m in bus.history[-20:]])
        inbox_text = "\n".join([f"{m.sender}: {m.content}" for m in inbox[-5:]])
        allowed = ", ".join(self.allowed_tools) or "(none)"
        prompt = f"""
Вы — {self.system_role} (имя агента: {self.name}).
Цель: {goal}

Входящие сообщения (последние):
{inbox_text}

История действий (последние):
{hist}

Вы можете выбрать ТОЛЬКО одно действие и должны вернуть JSON строго в соответствии с схемой:
AgentStep -> step: одно из ask_agent/use_tool/reply/finish.

- ask_agent: target в ['planner','coder','tester','reviewer'], message — сообщение
- use_tool: tool_name (должно быть одним из: {allowed}), args (dict) — аргументы инструмента
- reply: content (строка) — ответ
- finish: summary (строка) — итоговое сообщение

Если вы выбираете use_tool, убедитесь, что tool_name входит в ваш разрешённый набор. Если нет — используйте ask_agent для запроса разрешения.
Возвращайте ТОЛЬКО JSON. Никаких объяснений.
"""
        try:
            resp = call_llm_with_schema(prompt, AgentStep, system_prompt=self.system_role)
            return resp
        except Exception as e:
            Log.error(f"[{self.name}] LLM failed to produce valid AgentStep: {e}")
            step = {"step": {"action": "reply", "content": f"Error generating action: {str(e)[:200]}" }}
            return AgentStep.model_validate(step)


# ---------------------------
# Конкретные агенты
# ---------------------------

class PlannerAgent(AgentBase):
    """
    Агент планирования (planner).

    Задачи:
    -------
    - Определяет задачи и цели для других агентов.
    - Распределяет роли и инструменты.
    - Не использует инструменты Workspace напрямую.
    """
    def __init__(self):
        super().__init__("planner", "Planner: определяет задачи и назначает роли.", allowed_tools=[])


class CoderAgent(AgentBase):
    """
    Агент-программист (coder).

    Задачи:
    -------
    - Реализует функции и решения задачи.
    - Использует инструменты Workspace: store_code, read_code, run_python, lint_code.
    """
    def __init__(self):
        super().__init__(
            "coder",
            "Coder: реализует код с использованием инструментов рабочей среды.",
            allowed_tools=["store_code", "read_code", "run_python", "lint_code"]
        )


class TesterAgent(AgentBase):
    """
    Агент-тестировщик (tester).

    Задачи:
    -------
    - Генерирует и запускает тесты.
    - Использует инструменты Workspace: generate_tests, read_code, run_tests.
    """
    def __init__(self):
        super().__init__(
            "tester",
            "Tester: Генерирует и выполняет тесты.",
            allowed_tools=["generate_tests", "read_code", "run_tests"]
        )


class ReviewerAgent(AgentBase):
    """
    Агент-рецензент (reviewer).

    Задачи:
    -------
    - Проверяет качество кода, соблюдение стиля и best practices.
    - Может делать рекомендации и запросы на исправление.
    - Использует инструменты Workspace: read_code, lint_code, summarize_text.
    """
    def __init__(self):
        super().__init__(
            "reviewer",
            "Reviewer: Проверяет качество кода и запрашивает исправления.",
            allowed_tools=["read_code", "lint_code", "summarize_text"]
        )


In [19]:
# ---------------------------
# Оркестратор — управление агентами, контроль инструментов, логирование
# ---------------------------

class Orchestrator:
    """
    Оркестратор мультиагентной системы.

    Основная роль:
    ---------------
    - Управляет последовательностью действий агентов.
    - Контролирует использование инструментов (tools) и проверяет права доступа.
    - Логирует действия, ошибки и результаты.
    - Обеспечивает маршрутизацию сообщений между агентами через MessageBus.
    - Реализует правила завершения задачи (Finish).

    Атрибуты:
    ----------
    agents : Dict[str, AgentBase]
        Словарь всех агентов системы с ключами по имени роли.
        Примеры ключей: "planner", "coder", "tester", "reviewer".

    allowed_tools : Dict[str, List[str]]
        Словарь соответствий роли → список разрешённых инструментов.

    max_rounds : int
        Максимальное количество раундов взаимодействия агентов.
    """

    def __init__(self):
        self.agents = {
            "planner": PlannerAgent(),
            "coder": CoderAgent(),
            "tester": TesterAgent(),
            "reviewer": ReviewerAgent(),
        }
        self.allowed_tools = {name: agent.allowed_tools for name, agent in self.agents.items()}
        self.max_rounds = 10

    def step(self, goal: str, turn_order: List[str]) -> bool:
        """
        Выполняет один раунд действий всех агентов в заданном порядке.

        Параметры:
        -----------
        goal : str
            Текущая цель, которую система должна достичь.
        turn_order : List[str]
            Порядок обхода агентов в этом раунде.

        Логика работы:
        ---------------
        1. Для каждого агента в turn_order:
            - Получение сообщений из MessageBus (только для этого агента или broadcast).
            - Вызов метода agent.decide(goal, inbox) для выбора действия.
            - Логирование решения агента.

        2. Обработка действия агента:
            - AskAgent: пересылает сообщение указанному агенту.
            - UseTool: вызывает инструмент через Tools.call() при наличии прав, публикует результат в broadcast.
            - Reply: отправляет простое сообщение в broadcast.
            - Finish: публикует сообщение о завершении задачи и завершает оркестрацию.

        3. Если инструмент run_tests прошёл успешно, автоматически считается, что цель достигнута,
           и генерируется сообщение FINISH.

        Возвращает:
        -----------
        bool
            True, если задача достигнута и сгенерирован Finish, иначе False.
        """
        for name in turn_order:
            agent = self.agents[name]
            inbox = bus.receive_for(name)

            if inbox:
                Log.debug(f"[{name}] inbox ({len(inbox)}):")
                for m in inbox[-5:]:
                    Log.debug(f"  {m.sender}→{m.recipient}: {m.content[:200]}")

            try:
                decision = agent.decide(goal, inbox)
            except Exception as e:
                msg_text = f"Agent {name} failed to decide: {e}"
                Log.error(f"[orchestrator] {msg_text}")
                bus.send(BusMessage(sender="orchestrator", recipient=name, content=msg_text))
                continue

            Log.info(f"[{name}] decided: {decision.model_dump() if hasattr(decision,'model_dump') else decision}")

            step = decision.step

            # Обработка типа действия
            if isinstance(step, AskAgent):
                target = step.target
                bus.send(BusMessage(sender=name, recipient=target, content=step.message))
                Log.debug(f"[{name}] -> [ask->{target}]: {step.message[:200]}")

            elif isinstance(step, UseTool):
                tool = step.tool_name
                args = step.args or {}

                # Проверка прав на использование инструмента
                if tool not in self.allowed_tools.get(name, []):
                    msg = f"role {name} does not have access to tool {tool}. Allowed: {self.allowed_tools.get(name, [])}"
                    Log.warn(f"[orchestrator] DENIED: {msg}")
                    bus.send(BusMessage(sender="orchestrator", recipient=name, content=msg))
                    continue

                # Вызов инструмента
                try:
                    result = tools.call(tool, args)
                except Exception as e:
                    result = {"error": f"tool_exception: {e}\n{traceback.format_exc()}"}

                Log.info(f"[{name}] used {tool} -> {str(result)[:800]}")
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"tool {tool} result: {result}"))

                # Особая обработка: если run_tests прошёл успешно — завершение
                if tool == "run_tests" and isinstance(result, dict) and result.get("passed") is True:
                    summary = "Tests passed. Goal achieved."
                    bus.send(BusMessage(sender="orchestrator", recipient="broadcast", content=f"FINISH: {summary}"))
                    Log.info("[orchestrator] FINISH emitted.")
                    return True

            elif isinstance(step, Reply):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=step.content))
                Log.debug(f"[{name}] reply -> broadcast: {step.content[:200]}")

            elif isinstance(step, Finish):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"FINISH: {step.summary}"))
                Log.info(f"[{name}] finish -> broadcast: {step.summary[:200]}")
                return True

            else:
                msg = f"Unknown step type from agent {name}: {step}"
                Log.error("[orchestrator] " + msg)
                bus.send(BusMessage(sender="orchestrator", recipient=name, content=msg))

        return False

    def run(self, goal: str, order: Optional[List[str]] = None, max_rounds: int = 8):
        """
        Запускает цикл оркестрации до достижения цели или максимального количества раундов.

        Параметры:
        -----------
        goal : str
            Основная цель мультиагентной системы.
        order : Optional[List[str]]
            Порядок обхода агентов. По умолчанию: ["planner", "coder", "tester", "reviewer"].
        max_rounds : int
            Максимальное количество раундов (итераций взаимодействия агентов).

        Логика работы:
        ----------------
        1. Отправка стартового broadcast-сообщения от Planner.
        2. Итеративный вызов step() для каждого раунда.
        3. Если step() возвращает True, оркестрация считается завершённой.
        4. Если достигнут max_rounds — вывод сообщения о максимальном числе раундов.
        """
        if order is None:
            order = ["planner", "coder", "tester", "reviewer"]

        Log.info("=== ORCHESTRATION START ===")
        bus.send(BusMessage(sender="planner", recipient="broadcast", content=f"start: {goal}"))

        for r in range(1, max_rounds + 1):
            Log.info(f"\n--- ROUND {r} ---")
            finished = self.step(goal, order)
            if finished:
                Log.info("=== ORCHESTRATION FINISHED ===")
                return

        Log.warn("=== MAX ROUNDS REACHED ===")



In [20]:
bus = MessageBus()
ws = Workspace()
tools = Tools(ws)

In [21]:

# ---------------------------
# Demo scenario: implement fib and run tests
# ---------------------------
TESTS = """
assert fib(0) == 0
assert fib(1) == 1
assert fib(5) == 5
assert fib(10) == 55
print("ALL TESTS PASSED")
"""

goal = "Реализовать функцию bubble_sort(n: list) -> list, 0-indexed, итеративно без лишней памяти."

# insert a helpful initial planner broadcast (explicit instruction)
bus.send(BusMessage(sender="planner", recipient="broadcast",
                    content="Coder: используй инструмент use_tool/store_code для записи файла solution.py. Tester: используй generate_tests, затем run_tests. Reviewer: проведи lint и ревью кода."))

# Run orchestrator
orch = Orchestrator()
orch.run(goal, max_rounds=1)

# After run, print workspace artifacts and last few messages
print("\nWorkspace files:", list(ws.files.keys()))
if "solution.py" in ws.files:
    print("=== solution.py ===\n")
    print(ws.files["solution.py"])
print("\nLast bus history (tail 8):")
for m in bus.history[-8:]:
    print(f"  {m.sender} -> {m.recipient}: {str(m.content)[:200]}")


[DEBUG] planner → broadcast: Coder: используй инструмент use_tool/store_code для записи файла solution.py. Tester: используй generate_tests, затем run_tests. Reviewer: проведи lint и ревью кода.
[INFO] === ORCHESTRATION START ===
[DEBUG] planner → broadcast: start: Реализовать функцию bubble_sort(n: list) -> list, 0-indexed, итеративно без лишней памяти.
[INFO] 
--- ROUND 1 ---
[DEBUG] [planner] inbox (2):
[DEBUG]   planner→broadcast: Coder: используй инструмент use_tool/store_code для записи файла solution.py. Tester: используй generate_tests, затем run_tests. Reviewer: проведи lint и ревью кода.
[DEBUG]   planner→broadcast: start: Реализовать функцию bubble_sort(n: list) -> list, 0-indexed, итеративно без лишней памяти.
[DEBUG] [LLM] Вызов с prompt: 
Вы — Planner: определяет задачи и назначает роли. (имя агента: planner).
Цель: Реализовать функцию bubble_sort(n: list) -> list, 0-indexed, итеративно без лишней памяти.

Входящие сообщения (последни, system_prompt: Planner: определяет з